# Milestone 5 - Smart MCQ Solver Challenge
Complete notebook covering Questions 1-10.

Replace model checkpoint paths with your fine-tuned checkpoints before running.

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Base Models
DEBERTA_MODEL = "microsoft/deberta-v3-small"
ROBERTA_MODEL = "roberta-base"

# Load Tokenizers
tokenizer_deb = AutoTokenizer.from_pretrained(DEBERTA_MODEL)
tokenizer_rob = AutoTokenizer.from_pretrained(ROBERTA_MODEL)

# Load Models
model_deb = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_MODEL,
    num_labels=5,
    ignore_mismatched_sizes=True
).to(DEVICE).eval()

model_rob = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_MODEL,
    num_labels=5,
    ignore_mismatched_sizes=True
).to(DEVICE).eval()

# Load Data
train = pd.read_csv("/content/train (1).csv")
test = pd.read_csv("/content/test (1).csv")

label_map = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

def get_probs(model, tokenizer, text):
    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = F.softmax(logits, dim=-1).cpu().numpy()[0]
    return probs

def top3_string(prob):
    top3 = np.argsort(prob)[::-1][:3]
    return " ".join(label_map[i] for i in top3)

# Example
row = test.iloc[25]

text = row["prompt"] if "prompt" in test.columns else row.iloc[1]

deb_probs = get_probs(model_deb, tokenizer_deb, text)
rob_probs = get_probs(model_rob, tokenizer_rob, text)

print("DeBERTa:", deb_probs)
print("RoBERTa :", rob_probs)
print("Top-3 DeBERTa:", top3_string(deb_probs))
print("Top-3 RoBERTa :", top3_string(rob_probs))

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifie

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DeBERTa: [0.2281 0.1857 0.1798 0.1794 0.2269]
RoBERTa : [0.21666163 0.21263525 0.23718266 0.14417161 0.1893489 ]
Top-3 DeBERTa: A E B
Top-3 RoBERTa : C A B


## Q1

In [6]:
row=test.iloc[25]
text=row["prompt"] if "prompt" in test.columns else row.iloc[1]
p=get_probs(model_deb,tokenizer_deb,text)
idx=np.argmax(p)
print(label_map[idx],p[idx])

A 0.2281


## Q2

In [7]:
row=test.iloc[25]
text=row["prompt"] if "prompt" in test.columns else row.iloc[1]
p1=get_probs(model_deb,tokenizer_deb,text)
p2=get_probs(model_rob,tokenizer_rob,text)
avg=(p1+p2)/2
print(label_map[np.argmax(avg)])

A


## Q3

In [8]:
row=test.iloc[25]
text=row["prompt"] if "prompt" in test.columns else row.iloc[1]
p1=get_probs(model_deb,tokenizer_deb,text)
p2=get_probs(model_rob,tokenizer_rob,text)
ens=0.7*p1+0.3*p2
print(label_map[np.argmax(ens)])

A


## Q4

In [9]:
top3=np.argsort(ens)[::-1][:3]
print(" ".join(label_map[i] for i in top3))

A E C


## Q5

In [10]:
preds=[]
for _,row in test.iterrows():
    text=row["prompt"] if "prompt" in test.columns else row.iloc[1]
    p1=get_probs(model_deb,tokenizer_deb,text)
    p2=get_probs(model_rob,tokenizer_rob,text)
    ens=0.7*p1+0.3*p2
    top3=np.argsort(ens)[::-1][:3]
    preds.append(" ".join(label_map[i] for i in top3))
submission=pd.DataFrame({"id":test["id"],"prediction":preds})
submission.to_csv("submission.csv",index=False)
print(len(submission))

500


## Q6

In [11]:
instruction="Answer the following multiple-choice question carefully: "
changed=0
for i in range(50):
    row=test.iloc[i]
    text=row["prompt"] if "prompt" in test.columns else row.iloc[1]
    p=get_probs(model_deb,tokenizer_deb,text)
    p_aug=get_probs(model_deb,tokenizer_deb,instruction+text)
    tta=(p+p_aug)/2
    if np.argmax(p)!=np.argmax(tta):
        changed+=1
print(changed)

2


## Q7

In [12]:
changed=0
for i in range(100):
    row=test.iloc[i]
    text=row["prompt"] if "prompt" in test.columns else row.iloc[1]
    p1=get_probs(model_deb,tokenizer_deb,text)
    p2=get_probs(model_rob,tokenizer_rob,text)
    ens=0.7*p1+0.3*p2
    if np.argmax(p1)!=np.argmax(ens):
        changed+=1
print(changed)

1


## Q8

In [13]:
gain=0
for i in range(100):
    row=test.iloc[i]
    text=row["prompt"] if "prompt" in test.columns else row.iloc[1]
    p1=get_probs(model_deb,tokenizer_deb,text)
    p2=get_probs(model_rob,tokenizer_rob,text)
    ens=0.7*p1+0.3*p2
    if np.max(ens)>np.max(p1):
        gain+=1
print(gain)

0


## Q9

In [14]:
changed=0
for i in range(100):
    row=test.iloc[i]
    text=row["prompt"] if "prompt" in test.columns else row.iloc[1]
    p1=get_probs(model_deb,tokenizer_deb,text)
    p2=get_probs(model_rob,tokenizer_rob,text)
    ens=0.7*p1+0.3*p2
    t1=" ".join(label_map[j] for j in np.argsort(p1)[::-1][:3])
    t2=" ".join(label_map[j] for j in np.argsort(ens)[::-1][:3])
    if t1!=t2:
        changed+=1
print(changed)

100


## Q10

In [15]:
def apk(actual,pred,k=3):
    pred=pred[:k]
    for i,p in enumerate(pred):
        if p==actual:
            return 1/(i+1)
    return 0
def mapk(actuals,preds,k=3):
    return np.mean([apk(a,p,k) for a,p in zip(actuals,preds)])

# Replace 'answer' with the correct answer column if different.
actual=[]
preds=[]
val=train.iloc[:100]
for _,row in val.iterrows():
    p1=get_probs(model_deb,tokenizer_deb,row["prompt"])
    p2=get_probs(model_rob,tokenizer_rob,row["prompt"])
    ens=0.7*p1+0.3*p2
    top3=[label_map[j] for j in np.argsort(ens)[::-1][:3]]
    preds.append(top3)
    actual.append(row["answer"])
print(f"MAP@3 = {mapk(actual,preds):.4f}")

MAP@3 = 0.3267
